**Imports**

In [ ]:
import os
import torch
import numpy as np
import random
import numpy as np
import gymnasium
import ale_py

from utils import make_env, process_state
from dqn_agent import DQNAgent
from dqn_cnn_model import DQN_CNN_Model
from double_dqn_agent import DoubleDQNAgent

In [ ]:
SEED = 23

torch.manual_seed(SEED)
torch.backends.cudnn.deterministic=True # https://discuss.pytorch.org/t/what-is-the-differenc-between-cudnn-deterministic-and-cudnn-benchmark/38054
torch.backends.cudnn.benchmark=True # https://discuss.pytorch.org/t/what-does-torch-backends-cudnn-benchmark-do/5936/4
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"  
elif torch.backends.mps.is_available():
    DEVICE = "mps" 

In [ ]:
GRAY_SCALE = True 
SCREEN_SIZE = 84 
NUM_STACKED_FRAMES = 4 
SKIP_FRAMES = 4 
ENV_NAME = "ALE/Breakout-v5" 

## Entrenamiento

In [ ]:
#Hiperparámetros de entrenamiento del agente DQN
TOTAL_STEPS = 10_000_000
EPISODES = 10_000
STEPS_PER_EPISODE = 20_000

EPSILON_INI = 1
EPSILON_MIN = 0.05
EPSILON_ANNEAL_STEPS = 1_000_000

EPISODE_BLOCK = 100

BATCH_SIZE = 32
BUFFER_SIZE = 50_000

GAMMA = 0.995
LEARNING_RATE = 1e-5

In [ ]:
env = make_env(ENV_NAME,
                video_folder='./videos/dqn_training',
                name_prefix="breakout",
                record_every=500,
                grayscale=GRAY_SCALE,
                screen_size=SCREEN_SIZE,
                stack_frames=NUM_STACKED_FRAMES,
                skip_frames=SKIP_FRAMES
                )

net = DQN_CNN_Model(env.observation_space.shape, env.action_space.n).to(DEVICE)

dqn_agent = DQNAgent(env, net, process_state, BUFFER_SIZE, BATCH_SIZE, LEARNING_RATE, GAMMA, 
                     epsilon_i=EPSILON_INI, epsilon_f=EPSILON_MIN, 
                     epsilon_anneal_steps=EPSILON_ANNEAL_STEPS, 
                     episode_block=EPISODE_BLOCK, device=DEVICE)

rewards_dqn = dqn_agent.train(EPISODES, STEPS_PER_EPISODE, TOTAL_STEPS)

#save the rewards
np.save(os.path.join('rewards', 'dqn_rewards.npy'), rewards_dqn)

env.close()

In [ ]:
env = make_env(ENV_NAME,
                video_folder='./videos/ddqn_training',
                name_prefix="breakout",
                record_every=500,
                grayscale=GRAY_SCALE,
                screen_size=SCREEN_SIZE,
                stack_frames=NUM_STACKED_FRAMES,
                skip_frames=SKIP_FRAMES
                )

modelo_a = DQN_CNN_Model(env.observation_space.shape, env.action_space.n).to(DEVICE)
modelo_b = DQN_CNN_Model(env.observation_space.shape, env.action_space.n).to(DEVICE)

dqn_agent = DQNAgent(env, modelo_a, modelo_b, process_state, BUFFER_SIZE, BATCH_SIZE, LEARNING_RATE, GAMMA, 
                     epsilon_i=EPSILON_INI, epsilon_f=EPSILON_MIN, 
                     epsilon_anneal_steps=EPSILON_ANNEAL_STEPS, 
                     episode_block=EPISODE_BLOCK, device=DEVICE)

rewards_dqn = dqn_agent.train(EPISODES, STEPS_PER_EPISODE, TOTAL_STEPS)

#save the rewards
np.save(os.path.join('rewards', 'dqn_rewards.npy'), rewards_dqn)

env.close()

In [ ]:
# =================================================================
# ===== PASTE THIS AS THE ENTIRE LAST CELL OF YOUR NOTEBOOK ======
# =================================================================
import gymnasium

# --- 1. Vectorization Setup ---
NUM_ENVS = 48 # Number of parallel environments

# --- 2. Hyperparameters ---
TOTAL_STEPS = 1_000_000
BUFFER_SIZE = 100_000
BATCH_SIZE = 64
GAMMA = 0.99
LEARNING_RATE = 1e-4

EPSILON_INI = 1.0
EPSILON_MIN = 0.1
EPSILON_ANNEAL_STEPS = 1_000_000
EPISODE_BLOCK = 100


# --- 3. Create the Vectorized Environment (THE KEY FIX IS HERE) ---
def make_vec_env():
    env = make_env(ENV_NAME,
                    video_folder=None,
                    grayscale=GRAY_SCALE,
                    screen_size=SCREEN_SIZE,
                    stack_frames=NUM_STACKED_FRAMES,
                    skip_frames=SKIP_FRAMES
                   )
    return env

# This creates the final vectorized environment object
vec_env = gymnasium.vector.SyncVectorEnv([make_vec_env for _ in range(NUM_ENVS)])

# --- 5. Instantiate Agent and Model ---
# The corrected line
net = DQN_CNN_Model(vec_env.single_observation_space.shape, vec_env.single_action_space.n).to(DEVICE)


agent = DQNAgent(
    env=vec_env, 
    model=net, 
    obs_processing_func=process_state, 
    memory_buffer_size=BUFFER_SIZE, 
    batch_size=BATCH_SIZE, 
    learning_rate=LEARNING_RATE, 
    gamma=GAMMA, 
    epsilon_i=EPSILON_INI, 
    epsilon_f=EPSILON_MIN, 
    epsilon_anneal_steps=EPSILON_ANNEAL_STEPS, 
    episode_block=EPISODE_BLOCK, 
    device=DEVICE
)


# --- 6. Run the New Vectorized Training ---
# This will now run the corrected training loop and display all metrics
agent.train_vec(max_steps=TOTAL_STEPS)